# DocGen 文档生成语言 — 教程

DocGen 是基于 **Jinja2** 模板引擎的文档生成语言，专为 SysML 模型数据设计。
本教程将带你掌握完整的模板语法和 SysML 专用过滤器。

## 准备工作

先连接服务端并加载一个模型。

In [ ]:
from sysmldocgen_mdk.jupyter import register_magics
register_magics()

%connect http://localhost:8000 admin admin123
%load_model 2

## 1. 模板基础 — 变量替换

Jinja2 使用 `{{ variable }}` 输出变量值。

可用变量：
- `model_name` — 模型名称
- `model_version` — 模型版本
- `generate_time` — 渲染时间
- `element_count` — 元素总数
- `relationship_count` — 关系总数

In [ ]:
%%render_doc 基础变量
# {{ model_name }} 文档

| 属性 | 值 |
|------|----|
| 版本 | {{ model_version }} |
| 生成时间 | {{ generate_time }} |
| 元素数量 | {{ element_count }} |
| 关系数量 | {{ relationship_count }} |

## 2. 循环与条件

Jinja2 的 `{% for %}` 和 `{% if %}` 控制元素遍历。

每个元素有这些字段：
- `element_id` — 元素编号
- `element_name` — 元素名称
- `element_type` — 元素类型（Requirement, Block, Interface, Package...）
- `description` — 描述
- `attributes` — 附加属性（JSON 对象）

In [ ]:
%%render_doc 元素清单
# {{ model_name }} — 元素清单

## 所有 Block

{% for elem in elements if elem.element_type == "Block" %}
- **{{ elem.element_name }}** ({{ elem.element_id }})
  - 描述: {{ elem.description or "无" }}
{% else %}
_（无 Block 元素）_
{% endfor %}

## 所有需求 (Requirement)

{% for elem in elements if elem.element_type == "Requirement" %}
1. **{{ elem.element_name }}**: {{ elem.description or "无描述" }}
{% else %}
_（无需求元素）_
{% endfor %}

## 3. SysML 专用过滤器

DocGen 提供了 5 个专用过滤器，用管道符 `|` 调用。

### 3.1 req_table — 需求表格

将元素列表渲染为 HTML 表格，自动生成编号/名称/描述三列。

```
用法: {{ elements | req_table("标题") }}
```

In [ ]:
%%render_doc 需求表格
# 需求规格 — 表格视图

{{ elements | req_table("完整元素列表") }}

### 3.2 block_hierarchy — Block 层级树

按 `parent_element_id` 自动构建层级树，展示系统结构。
只筛选 element_type 为 "Block" 的元素。

```
用法: {{ elements | block_hierarchy }}
```

In [ ]:
%%render_doc Block层级
# 系统架构 — Block 层级

{{ elements | block_hierarchy }}

### 3.3 iface_matrix — 接口矩阵

筛选 Interface 类型的元素，生成接口矩阵表格。

```
用法: {{ elements | iface_matrix }}
```

In [ ]:
%%render_doc 接口矩阵
# 接口定义

{{ elements | iface_matrix }}

### 3.4 format_attrs — 格式化属性

将 JSON 格式的 attributes 渲染为可读的列表。

```
用法: {{ elem.attributes | format_attrs }}
```

In [ ]:
%%render_doc 元素属性
# 元素属性详情

{% for elem in elements[:5] %}
### {{ elem.element_name }}
- **类型**: {{ elem.element_type }}
- **属性**: {{ elem.attributes | format_attrs }}

{% endfor %}

### 3.5 element_link — 交叉引用链接

生成指向模型元素的 HTML 锚点链接。

```
用法: {{ "req_1" | element_link("查看详情") }}
```

In [ ]:
%%render_doc 交叉引用
# 元素引用

快捷链接：
- 第一个元素：{{ model_2_elements.iloc[0].element_id | element_link }}
- 第二个元素：{{ model_2_elements.iloc[1].element_id | element_link("点我查看") }}

## 4. 类型快捷变量

系统自动为每种元素类型生成快捷变量：`{{ requirement_list }}`, `{{ block_list }}` 等。

可直接在循环中使用。

In [ ]:
%%render_doc 类型快捷变量
# 快捷变量演示

可用类型：
{% for elem in elements[:3] %}
- {{ elem.element_type }} → `{{ elem.element_type.lower() }}_list`
{% endfor %}

## 5. 综合示例 — 完整文档模板

综合运用所有功能，生成一份完整的需求规格说明书。

In [ ]:
%%render_doc 综合示例：智能家居系统需求规格
<h1 style="text-align:center">{{ model_name }} 需求规格说明书</h1>
<p style="text-align:center;color:#666">
  版本 {{ model_version }} | 生成时间 {{ generate_time }}
</p>

---

## 1. 文档概述

本文档基于 SysML 模型 **{{ model_name }}** 自动生成，
共包含 {{ element_count }} 个模型元素和 {{ relationship_count }} 条关系。

## 2. 需求规格

{{ requirement_list | req_table("系统需求清单") }}

## 3. 系统架构

### 3.1 Block 层级

{{ elements | block_hierarchy }}

### 3.2 接口定义

{{ elements | iface_matrix }}

---

> 本文档由 SysMLDocGen 系统自动生成

## 6. 推送到服务端

将写好的模板保存到服务端，供团队使用或用于批量文档生成。

In [ ]:
from sysmldocgen_mdk import DocGenTemplate, Client

tpl = DocGenTemplate(
    name="智能家居需求规格",
    content="""<h1>{{ model_name }} 需求规格说明书</h1>
<p>版本 {{ model_version }} | {{ generate_time }}</p>

## 需求
{{ requirement_list | req_table("需求清单") }}

## 架构
{{ elements | block_hierarchy }}
"""
)

client = Client("http://localhost:8000", "admin", "admin123")
result = tpl.to_server(client, template_type="docgen")
print(f"模板已保存: ID={result['id']}, 名称={result['name']}")

---

## 总结

DocGen 模板语言的核心能力：

| 能力 | 语法 | 用途 |
|------|------|------|
| 变量替换 | `{{ var }}` | 输出模型名称、版本等 |
| 循环 | `{% for %}` | 遍历元素列表 |
| 条件 | `{% if %}` | 按类型筛选元素 |
| 需求表格 | `\| req_table` | 生成需求清单表格 |
| Block 层级 | `\| block_hierarchy` | 展示系统结构树 |
| 接口矩阵 | `\| iface_matrix` | 列出接口定义 |
| 属性格式化 | `\| format_attrs` | 显示 JSON 属性 |
| 交叉引用 | `\| element_link` | 元素间跳转链接 |

下一步：在 `demo.ipynb` 中通过服务端模板生成最终文档。